In [3]:
# ========== 导入库 ==========
import os
import sys
import time
import warnings
from datetime import datetime

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa

warnings.filterwarnings('ignore')

print('所有库导入成功！')

# 路径设置
PROJ_DIR = os.path.abspath(os.getcwd())
DATA_DIR = os.path.join(PROJ_DIR, 'data')

# 股票信息
STOCKS = {
    '002142': ('宁波银行', '银行'),
    '600036': ('招商银行', '银行'),
    '002594': ('比亚迪', '汽车'),
    '600048': ('保利发展', '房地产'),
    '000063': ('中兴通讯', '通讯'),
    '600487': ('亨通光电', '通讯'),
    '600519': ('贵州茅台', '白酒'),
    '300750': ('宁德时代', '能源'),
    '600900': ('长江电力', '能源'),
    '002352': ('顺丰控股', '物流'),
}

print('库加载完成，路径已设置')

所有库导入成功！
库加载完成，路径已设置


# 02 - 数据清洗与存储

本 Notebook 负责对下载的原始数据进行清洗、转换和合并，包括：
1. 单表清洗（缺失值、日期格式、数据类型、重复值、离群值）
2. 宽表与长表转换
3. 多表合并（股票 + 指数 + 宏观数据）
4. 进阶存储：Parquet 格式（方式 B）与 CSV 对比

---
## 3.1 单表清洗

对每只股票的原始数据依次进行 6 项清洗操作。以下以循环方式逐只处理。

In [4]:
# ========== 3.1 单表清洗 ==========

# 存储清洗后的每只股票数据
cleaned_stocks = {}

for code, (name, industry) in STOCKS.items():
    print(f'\n{"="*60}')
    print(f'清洗: {name} ({code}) - {industry}')
    print(f'{"="*60}')
    
    # 读取原始 CSV
    file_path = os.path.join(DATA_DIR, 'stock', f'stock_{code}.csv')
    if not os.path.exists(file_path):
        print(f'  [跳过] 文件不存在: {file_path}')
        continue
    
    df = pd.read_csv(file_path)
    print(f'  原始数据形状: {df.shape}')
    
    # ---------- ① 缺失值检测 ----------
    print('\n  --- ① 缺失值检测 ---')
    missing_count = df.isnull().sum()
    missing_pct = df.isnull().sum() / len(df) * 100
    missing_df = pd.DataFrame({'缺失数量': missing_count, '缺失比例(%)': missing_pct})
    missing_df = missing_df[missing_df['缺失数量'] > 0]
    if len(missing_df) > 0:
        print(f'  发现缺失值列:')
        print(missing_df.to_string())
        print('  - 缺失可能原因：停牌期间无交易、节假日非交易日、数据源接口不稳定等')
    else:
        print('  无缺失值')
    
    # ---------- ② 缺失值处理 ----------
    print('\n  --- ② 缺失值处理 ---')
    before_ffill = df.isnull().sum().sum()
    # 对价格列使用 ffill，若仍缺失则删除
    price_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
    price_cols = [c for c in price_cols if c in df.columns]
    df[price_cols] = df[price_cols].ffill()
    # 删除仍然缺失的行
    df.dropna(subset=price_cols, inplace=True)
    after_ffill = df.isnull().sum().sum()
    print(f'  缺失值处理前: {before_ffill} 个缺失')
    print(f'  采用向前填充(ffill)，处理后: {after_ffill} 个缺失')
    print(f'  选择 ffill 的依据：金融时间序列中，停牌日的价格可参考上一交易日，符合市场惯例')
    
    # ---------- ③ 日期格式统一 ----------
    print('\n  --- ③ 日期格式统一 ---')
    print(f'  原始日期类型: {df["date"].dtype}')
    df['date'] = pd.to_datetime(df['date'])
    print(f'  转换后日期类型: {df["date"].dtype}')
    df.set_index('date', inplace=True)
    print(f'  日期已设为索引，日期范围: {df.index.min()} ~ {df.index.max()}')
    
    # ---------- ④ 数据类型检查 ----------
    print('\n  --- ④ 数据类型检查 ---')
    numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
    numeric_cols = [c for c in numeric_cols if c in df.columns]
    for col in numeric_cols:
        if df[col].dtype == 'object':
            print(f'  {col}: 原为 {df[col].dtype}，转换为数值型')
            df[col] = pd.to_numeric(df[col], errors='coerce')
        else:
            print(f'  {col}: {df[col].dtype} ✓')
    
    # ---------- ⑤ 重复值处理 ----------
    print('\n  --- ⑤ 重复值处理 ---')
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        df.drop_duplicates(inplace=True)
        print(f'  删除 {dup_count} 行重复数据')
    else:
        print('  无重复行')
    
    # ---------- ⑥ 离群值标注 ----------
    print('\n  --- ⑥ 离群值标注 ---')
    # 计算日收益率
    df['return'] = np.log(df['close'] / df['close'].shift(1))
    df['is_extreme'] = False
    extreme_mask = df['return'].abs() > 0.20
    df.loc[extreme_mask, 'is_extreme'] = True
    extreme_count = extreme_mask.sum()
    print(f'  日收益率计算完成')
    print(f'  标注为极端值的交易日: {extreme_count} 天')
    if extreme_count > 0:
        extreme_dates = df[extreme_mask].index.strftime('%Y-%m-%d').tolist()
        print(f'  极端值日期: {extreme_dates}')
        print(f'  可能成因：重大利好/利空公告、分红除权除息日、重组复牌等')
    
    # 存储清洗后的数据
    df.reset_index(inplace=True)
    df['code'] = code
    df['name'] = name
    df['industry'] = industry
    cleaned_stocks[code] = df
    print(f'\n  清洗完成! 最终形状: {df.shape}')

print('\n' + '='*60)
print('所有股票清洗完成！')


清洗: 宁波银行 (002142) - 银行
  原始数据形状: (1545, 7)

  --- ① 缺失值检测 ---
  发现缺失值列:
        缺失数量  缺失比例(%)
volume     6  0.38835
amount     6  0.38835
  - 缺失可能原因：停牌期间无交易、节假日非交易日、数据源接口不稳定等

  --- ② 缺失值处理 ---
  缺失值处理前: 12 个缺失
  采用向前填充(ffill)，处理后: 0 个缺失
  选择 ffill 的依据：金融时间序列中，停牌日的价格可参考上一交易日，符合市场惯例

  --- ③ 日期格式统一 ---
  原始日期类型: str
  转换后日期类型: datetime64[us]
  日期已设为索引，日期范围: 2020-01-02 00:00:00 ~ 2026-05-22 00:00:00

  --- ④ 数据类型检查 ---
  open: float64 ✓
  high: float64 ✓
  low: float64 ✓
  close: float64 ✓
  volume: float64 ✓
  amount: float64 ✓

  --- ⑤ 重复值处理 ---
  删除 5 行重复数据

  --- ⑥ 离群值标注 ---
  日收益率计算完成
  标注为极端值的交易日: 0 天

  清洗完成! 最终形状: (1540, 12)

清洗: 招商银行 (600036) - 银行
  原始数据形状: (1545, 7)

  --- ① 缺失值检测 ---
  无缺失值

  --- ② 缺失值处理 ---
  缺失值处理前: 0 个缺失
  采用向前填充(ffill)，处理后: 0 个缺失
  选择 ffill 的依据：金融时间序列中，停牌日的价格可参考上一交易日，符合市场惯例

  --- ③ 日期格式统一 ---
  原始日期类型: str
  转换后日期类型: datetime64[us]
  日期已设为索引，日期范围: 2020-01-02 00:00:00 ~ 2026-05-22 00:00:00

  --- ④ 数据类型检查 ---
  open: float64 ✓
  high: float64 ✓
  low: fl

In [5]:
# ========== 合并所有清洗后的股票数据 ==========
if cleaned_stocks:
    df_all_stocks = pd.concat(cleaned_stocks.values(), ignore_index=True)
    print(f'合并后全部股票数据: {df_all_stocks.shape}')
    print(f'列名: {df_all_stocks.columns.tolist()}')
else:
    print('错误: 没有清洗后的股票数据')

合并后全部股票数据: (15434, 12)
列名: ['date', 'open', 'close', 'high', 'low', 'volume', 'amount', 'return', 'is_extreme', 'code', 'name', 'industry']


---
## 3.2 宽表与长表转换

将 10 只股票的收盘价合并为宽表，再转换为长表，并讨论两种格式的适用场景。

In [6]:
# ========== 3.2 宽表与长表转换 ==========

# 创建宽表：日期为索引，每列一只股票的收盘价
wide_df = df_all_stocks.pivot_table(
    index='date',
    columns='code',
    values='close',
    aggfunc='first'
)

print('=== 宽表（收盘价）===')
print(f'形状: {wide_df.shape}')
print(wide_df.head())

print('\n' + '='*60)

# 转换回长表
long_df = wide_df.reset_index().melt(
    id_vars='date',
    var_name='code',
    value_name='close'
)

print('=== 长表（收盘价）===')
print(f'形状: {long_df.shape}')
print(long_df.head())

print('\n' + '='*60)
print('''
【宽表 vs 长表适用场景】

宽表（Wide format）适合：
- 计算股票间的相关系数矩阵
- 时间序列分析（如直接对每列计算收益率）
- 可视化多只股票的价格走势（每列作为一个序列）
- 方便查看某个交易日所有股票的价格

长表（Long format）适合：
- 使用 groupby 按股票分组统计
- 后续多表合并（可按 code 字段关联）
- 在 seaborn/matplotlib 中进行分面绘图
- 机器学习/回归分析（每行一个样本）
- 数据库存储（符合第三范式）
''')

=== 宽表（收盘价）===
形状: (1545, 10)
code           000063     002142     002352     002594     300750     600036  \
date                                                                           
2020-01-02  32.718117  24.260699  33.969811  15.582369  55.253130  29.432976   
2020-01-03  33.733347  24.344069  33.706971  15.540315  56.296321  29.826627   
2020-01-06  33.871788  24.402428  33.398813  15.617952  56.059933  29.705504   
2020-01-07  33.779494  24.519146  34.169207  15.543550  55.756740  29.637372   
2020-01-08  32.958080  24.093959  33.761351  15.294465  56.322015  29.077177   

code           600048     600487      600519     600900  
date                                                     
2020-01-02  12.602703  15.766334  979.045560  14.995442  
2020-01-03  12.362430  15.708932  934.477327  15.076630  
2020-01-06  12.153160  15.814168  933.983472  14.776234  
2020-01-07  12.238418  16.359485  948.313926  14.768115  
2020-01-08  12.029148  16.149012  942.777554  14.654452  

==

---
## 3.3 多表合并

将个股数据与指数数据、宏观数据按日期合并。

In [7]:
# ========== 3.3.1 加载指数数据 ==========

def load_index(code):
    """加载指数 CSV 数据"""
    path = os.path.join(DATA_DIR, 'index', f'index_{code}.csv')
    if not os.path.exists(path):
        print(f'[警告] 指数文件不存在: {path}')
        return None
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date'])
    df.rename(columns={'close': f'index_{code}_close'}, inplace=True)
    return df[['date', f'index_{code}_close']]

df_idx_300 = load_index('000300')
df_idx_001 = load_index('000001')

print(f'沪深300数据: {df_idx_300.shape if df_idx_300 is not None else "None"}')
print(f'上证指数数据: {df_idx_001.shape if df_idx_001 is not None else "None"}')

沪深300数据: (1545, 2)
上证指数数据: (1545, 2)


In [8]:
# ========== 3.3.2 加载宏观数据 ==========

def load_macro_cpi():
    """加载 CPI 数据（macro_china_cpi 格式）"""
    path = os.path.join(DATA_DIR, 'macro', 'macro_cpi.csv')
    if not os.path.exists(path):
        print(f'[警告] CPI文件不存在: {path}')
        return None
    df = pd.read_csv(path)
    print(f'CPI列名: {df.columns.tolist()}')
    # 提取月份和全国同比增长率
    result = df[['月份', '全国-同比增长']].copy()
    result.columns = ['date', 'macro_cpi']
    result['date'] = result['date'].str.replace('年', '-').str.replace('月份', '')
    result['date'] = pd.to_datetime(result['date'])
    result['macro_cpi'] = pd.to_numeric(result['macro_cpi'], errors='coerce')
    return result

def load_macro_m2():
    """加载 M2 数据（macro_china_money_supply 格式）"""
    path = os.path.join(DATA_DIR, 'macro', 'macro_m2.csv')
    if not os.path.exists(path):
        print(f'[警告] M2文件不存在: {path}')
        return None
    df = pd.read_csv(path)
    print(f'M2列名: {df.columns.tolist()}')
    # 提取月份和 M2 同比增长
    result = df[['月份', '货币和准货币(M2)-同比增长']].copy()
    result.columns = ['date', 'macro_m2']
    result['date'] = result['date'].str.replace('年', '-').str.replace('月份', '')
    result['date'] = pd.to_datetime(result['date'])
    result['macro_m2'] = pd.to_numeric(result['macro_m2'], errors='coerce')
    return result

df_cpi = load_macro_cpi()
df_m2 = load_macro_m2()

if df_cpi is not None:
    print(f'\nCPI 数据预览:')
    print(df_cpi.head())
if df_m2 is not None:
    print(f'\nM2 数据预览:')
    print(df_m2.head())

CPI列名: ['月份', '全国-当月', '全国-同比增长', '全国-环比增长', '全国-累计', '城市-当月', '城市-同比增长', '城市-环比增长', '城市-累计', '农村-当月', '农村-同比增长', '农村-环比增长', '农村-累计']
M2列名: ['月份', '货币和准货币(M2)-数量(亿元)', '货币和准货币(M2)-同比增长', '货币和准货币(M2)-环比增长', '货币(M1)-数量(亿元)', '货币(M1)-同比增长', '货币(M1)-环比增长', '流通中的现金(M0)-数量(亿元)', '流通中的现金(M0)-同比增长', '流通中的现金(M0)-环比增长']

CPI 数据预览:
        date  macro_cpi
0 2026-04-01        1.2
1 2026-03-01        1.0
2 2026-02-01        1.3
3 2026-01-01        0.2
4 2025-12-01        0.8

M2 数据预览:
        date  macro_m2
0 2026-04-01       8.6
1 2026-03-01       8.5
2 2026-02-01       9.0
3 2026-01-01       9.0
4 2025-12-01       8.5


In [9]:
# ========== 3.3.3 合并数据 ==========

# Step 1: 个股数据与指数数据 left join
print('=== 合并个股与指数数据 ===')
print(f'合并前个股数据行数: {len(df_all_stocks)}')

df_merged = df_all_stocks.copy()
df_merged['date'] = pd.to_datetime(df_merged['date'])

if df_idx_300 is not None:
    df_merged = df_merged.merge(df_idx_300, on='date', how='left')
if df_idx_001 is not None:
    df_merged = df_merged.merge(df_idx_001, on='date', how='left')

print(f'合并后数据行数: {len(df_merged)}')
print(f'合并后列数: {len(df_merged.columns)}')
print(f'新增列: {[c for c in df_merged.columns if "index_" in c]}')

# Step 2: 合并月度宏观数据（将月度数据映射到对应月份的每个交易日）
print('\n=== 合并宏观数据 ===')

# 创建年月辅助列
df_merged['year_month'] = df_merged['date'].dt.strftime('%Y-%m')

if df_cpi is not None:
    df_cpi['year_month'] = df_cpi['date'].dt.strftime('%Y-%m')
    before = len(df_merged)
    df_merged = df_merged.merge(
        df_cpi[['year_month', 'macro_cpi']],
        on='year_month',
        how='left'
    )
    print(f'合并CPI后行数: {before} -> {len(df_merged)}')

if df_m2 is not None:
    df_m2['year_month'] = df_m2['date'].dt.strftime('%Y-%m')
    before = len(df_merged)
    df_merged = df_merged.merge(
        df_m2[['year_month', 'macro_m2']],
        on='year_month',
        how='left'
    )
    print(f'合并M2后行数: {before} -> {len(df_merged)}')

# 删除辅助列
df_merged.drop(columns=['year_month'], inplace=True)

print(f'\n最终合并数据形状: {df_merged.shape}')
print(f'列名: {df_merged.columns.tolist()}')
print('\n行数变化原因说明：')
print('- left join 不会改变左表的行数（个股数据为主表）')
print('- 宏观数据从月度频率映射到日度频率，每月的所有交易日共享同一个月度宏观值')
print('- 如果某只股票在某日期没有交易数据（停牌），则该日期不会出现在个股数据中')

=== 合并个股与指数数据 ===
合并前个股数据行数: 15434
合并后数据行数: 15434
合并后列数: 14
新增列: ['index_000300_close', 'index_000001_close']

=== 合并宏观数据 ===
合并CPI后行数: 15434 -> 15434
合并M2后行数: 15434 -> 15434

最终合并数据形状: (15434, 16)
列名: ['date', 'open', 'close', 'high', 'low', 'volume', 'amount', 'return', 'is_extreme', 'code', 'name', 'industry', 'index_000300_close', 'index_000001_close', 'macro_cpi', 'macro_m2']

行数变化原因说明：
- left join 不会改变左表的行数（个股数据为主表）
- 宏观数据从月度频率映射到日度频率，每月的所有交易日共享同一个月度宏观值
- 如果某只股票在某日期没有交易数据（停牌），则该日期不会出现在个股数据中


In [10]:
# ========== 保存合并后的综合数据 ==========
# 方式 A：CSV
combined_csv_path = os.path.join(DATA_DIR, 'combined', 'combined_data.csv')
df_merged.to_csv(combined_csv_path, index=False, encoding='utf-8-sig')
print(f'综合数据已保存至: {combined_csv_path}')
print(f'文件大小: {os.path.getsize(combined_csv_path)/1024:.1f} KB')

综合数据已保存至: e:\lianyujun_homework\my own\dshw-p01\data\combined\combined_data.csv
文件大小: 2473.6 KB


---
## 方式 B：Parquet 进阶存储

将清洗后的数据额外保存为 Parquet 格式，并对比 CSV 与 Parquet 的读取速度和文件体积。

In [11]:
# ========== 方式 B：Parquet 存储 ==========

# 保存为 Parquet
parquet_path = os.path.join(DATA_DIR, 'clean', 'stock_clean.parquet')
df_merged.to_parquet(parquet_path, index=False)
print(f'Parquet 文件已保存至: {parquet_path}')

# 同时也保存一份 CSV 到 clean 目录用于对比
clean_csv_path = os.path.join(DATA_DIR, 'clean', 'stock_clean.csv')
df_merged.to_csv(clean_csv_path, index=False, encoding='utf-8-sig')
print(f'CSV 文件已保存至: {clean_csv_path}')

print('\n' + '='*60)
print('=== Parquet 特性演示 ===')

# ① 列式读取
print('\n① 列式读取（只加载部分列）:')
df_partial = pd.read_parquet(parquet_path, columns=['date', 'code', 'close'])
print(f'  读取 3 列: {df_partial.shape}')
print(df_partial.head())

# ② 查看 Schema
print('\n② Schema（类型契约）:')
schema = pq.read_schema(parquet_path)
print(schema)

# ③ 读取速度与文件体积对比
print('\n③ CSV vs Parquet 性能对比:')

# CSV 读取
t0 = time.time()
pd.read_csv(clean_csv_path)
csv_time = time.time() - t0
csv_size = os.path.getsize(clean_csv_path) / 1024

# Parquet 读取
t0 = time.time()
pd.read_parquet(parquet_path)
parquet_time = time.time() - t0
parquet_size = os.path.getsize(parquet_path) / 1024

print(f'CSV  读取耗时: {csv_time:.3f}s  文件大小: {csv_size:.1f} KB')
print(f'Parquet 读取耗时: {parquet_time:.3f}s  文件大小: {parquet_size:.1f} KB')
print(f'速度比: {csv_time/parquet_time:.1f}x' if parquet_time > 0 else '')
print(f'体积比: {csv_size/parquet_size:.1f}x')

print('\n【分析】')
print('在本次数据规模下（约数十万行），CSV 与 Parquet 的读取时间差异可能不明显，')
print('但 Parquet 通常体积更小（列式压缩存储）。在大规模数据（数百万行以上）场景下，')
print('Parquet 的列式读取优势会非常显著，尤其是只需读取少数列进行分析时。')

Parquet 文件已保存至: e:\lianyujun_homework\my own\dshw-p01\data\clean\stock_clean.parquet
CSV 文件已保存至: e:\lianyujun_homework\my own\dshw-p01\data\clean\stock_clean.csv

=== Parquet 特性演示 ===

① 列式读取（只加载部分列）:
  读取 3 列: (15434, 3)
        date    code      close
0 2020-01-02  002142  24.260699
1 2020-01-03  002142  24.344069
2 2020-01-06  002142  24.402428
3 2020-01-07  002142  24.519146
4 2020-01-08  002142  24.093959

② Schema（类型契约）:
date: timestamp[us]
open: double
close: double
high: double
low: double
volume: double
amount: double
return: double
is_extreme: bool
code: large_string
name: large_string
industry: large_string
index_000300_close: double
index_000001_close: double
macro_cpi: double
macro_m2: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1930

③ CSV vs Parquet 性能对比:
CSV  读取耗时: 0.047s  文件大小: 2473.6 KB
Parquet 读取耗时: 0.007s  文件大小: 1029.3 KB
速度比: 6.7x
体积比: 2.4x

【分析】
在本次数据规模下（约数十万行），CSV 与 Parquet 的读取时间差异可能不明显，
但 Parquet 通常体积

In [12]:
# ========== 最终验证 ==========
print('=== 清洗后数据文件 ===')
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f.endswith('.csv') or f.endswith('.parquet'):
            fp = os.path.join(root, f)
            size_kb = os.path.getsize(fp) / 1024
            print(f'  {fp}  ({size_kb:.1f} KB)')

print(f'\n清洗后综合数据行数: {len(df_merged)}')
print(f'日期范围: {df_merged["date"].min()} ~ {df_merged["date"].max()}')
print(f'股票数量: {df_merged["code"].nunique()}')

=== 清洗后数据文件 ===
  e:\lianyujun_homework\my own\dshw-p01\data\clean\stock_clean.csv  (2473.6 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\clean\stock_clean.parquet  (1029.3 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\combined\combined_data.csv  (2473.6 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\finance\finance_ratios.csv  (19.2 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\index\index_000001.csv  (95.8 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\index\index_000300.csv  (95.6 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\macro\macro_cpi.csv  (20.7 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\macro\macro_m2.csv  (21.4 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_000063.csv  (117.0 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_002142.csv  (122.9 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_002352.csv  (121.3 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_002594.csv  (117.8 KB)
  e:\lianyujun_homework